## 1. Importation & Nettoyage des Données

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

# Chargement
df = pd.read_csv('global_power_plant_database.csv')
print(f"Dimensions brutes : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")


In [ ]:

# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Manquants': missing, '%': missing_pct}).sort_values('%', ascending=False).head(12)


In [ ]:

# Conversion NumPy des colonnes numériques
for col in ['capacity_mw', 'latitude', 'longitude', 'commissioning_year']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Compléter la production réelle manquante avec les estimations
for y in [2013, 2014, 2015, 2016, 2017]:
    col_a = f'generation_gwh_{y}'; col_e = f'estimated_generation_gwh_{y}'
    if col_a in df.columns and col_e in df.columns:
        df[col_a].fillna(df[col_e], inplace=True)

# Nettoyage des lignes sans données essentielles
df.dropna(subset=['primary_fuel', 'capacity_mw'], inplace=True)
print(f"Dimensions après nettoyage : {df.shape}")
print(f"Types de combustible uniques ({df['primary_fuel'].nunique()}) : {sorted(df['primary_fuel'].unique())}")


## 2. Analyse Exploratoire des Données (EDA)

In [ ]:
# Statistiques descriptives
gen_cols = [f'generation_gwh_{y}' for y in [2013,2014,2015,2016,2017]]
df[['capacity_mw'] + gen_cols].describe().round(2)


In [ ]:

top_countries = df.groupby('country_long')['capacity_mw'].sum().sort_values(ascending=False).head(15)
top_fuels = df['primary_fuel'].value_counts().head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_countries.plot(kind='barh', ax=axes[0], color=sns.color_palette("Blues_r", 15))
axes[0].set_title("Top 15 pays – Capacité totale installée (MW)", fontweight='bold')
axes[0].set_xlabel("Capacité (MW)"); axes[0].invert_yaxis()

# Combustibles les plus utilisés
top_fuels.plot(kind='bar', ax=axes[1], color=sns.color_palette("Set2", 12))
axes[1].set_title("Nb de centrales par type de combustible", fontweight='bold')
axes[1].set_xlabel("Combustible"); axes[1].set_ylabel("Nb centrales")
axes[1].tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()
# Observation : USA, Chine, Inde dominent. Le solaire est le fuel le plus représenté en nombre.


## 3. Analyse Statistique – Puissance par Type de Carburant

In [ ]:
# Analyse par type de combustible
fuels_main = df['primary_fuel'].value_counts().head(8).index.tolist()
df_main = df[df['primary_fuel'].isin(fuels_main)]

fuel_stats = df_main.groupby('primary_fuel')['capacity_mw'].agg(
    Moyenne='mean', Médiane='median', Écart_type='std', Nb_centrales='count').round(2)
print(fuel_stats.sort_values('Moyenne', ascending=False).to_string())


In [ ]:

# ANOVA – test si les moyennes diffèrent entre carburants
groups = [df_main[df_main['primary_fuel']==f]['capacity_mw'].dropna().values for f in fuels_main]
f_stat, p_val = stats.f_oneway(*groups)
print(f"ANOVA : F = {f_stat:.2f},  p-value = {p_val:.2e}")
print("✅ Différences SIGNIFICATIVES (p < 0.05) entre types de carburant" if p_val < 0.05 else "❌ Pas de différence significative")

# Tests t pairwise (Welch) qui déterminent quelles paires de carburants diffèrent significativement
from itertools import combinations
results = []
for f1, f2 in combinations(fuels_main, 2):
    g1 = df_main[df_main['primary_fuel']==f1]['capacity_mw'].dropna()
    g2 = df_main[df_main['primary_fuel']==f2]['capacity_mw'].dropna()
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    results.append({'Paire': f'{f1} vs {f2}', 't': round(t,2), 'p': round(p,4)})
pd.DataFrame(results).sort_values('p').head(8)


In [ ]:
# Visualisation des distributions de capacité par type de combustible
fig, ax = plt.subplots(figsize=(14, 5))
order = df_main.groupby('primary_fuel')['capacity_mw'].median().sort_values(ascending=False).index
sns.boxplot(data=df_main, x='primary_fuel', y='capacity_mw', order=order,
            palette='Set3', showfliers=False, ax=ax)
ax.set_title("Distribution de la capacité (MW) par type de combustible\n(valeurs aberrantes masquées)", fontweight='bold')
ax.set_xlabel("Type de combustible"); ax.set_ylabel("Capacité (MW)")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()


## 4. Analyse des Séries Chronologiques

In [ ]:

df_year = df.dropna(subset=['commissioning_year'])
df_year = df_year[(df_year['commissioning_year'] >= 1900) & (df_year['commissioning_year'] <= 2020)].copy()
df_year['commissioning_year'] = df_year['commissioning_year'].astype(int)

# Tendance de mise en service des centrales dans le temps
yearly = df_year.groupby('commissioning_year').size().reset_index(name='count')
x, y = yearly['commissioning_year'].values, yearly['count'].values
coef = np.polyfit(x, y, 1)          # NumPy : régression linéaire
trend = np.poly1d(coef)
print(f"Tendance : +{coef[0]:.2f} nouvelles centrales par an en moyenne")


fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x, y, color='steelblue', alpha=0.55, label='Centrales/an')
ax.plot(x, trend(x), 'r--', lw=2.5, label=f'Tendance linéaire (+{coef[0]:.1f}/an)')
ax.set_title("Nombre de centrales mises en service par année", fontweight='bold')
ax.set_xlabel("Année"); ax.set_ylabel("Nombre de centrales")
ax.set_xlim(1900, 2020); ax.legend(); plt.tight_layout(); plt.show()


In [ ]:

df_year['decade'] = (df_year['commissioning_year'] // 10) * 10
top6_fuel = df_year['primary_fuel'].value_counts().head(6).index
fuel_decade = df_year.groupby(['decade','primary_fuel']).size().unstack(fill_value=0)
fuel_decade_pct = fuel_decade[top6_fuel].div(fuel_decade[top6_fuel].sum(axis=1), axis=0) * 100

# Visualisation de l'évolution de la composition énergétique par décennie
fuel_decade_pct.plot(kind='area', figsize=(13,5), colormap='tab10', alpha=0.72)
plt.title("Évolution de la composition énergétique par décennie (%)", fontweight='bold')
plt.xlabel("Décennie"); plt.ylabel("Part relative (%)")
plt.legend(loc='upper left', fontsize=9); plt.tight_layout(); plt.show()
# Observation : montée en puissance spectaculaire du solaire depuis 2010.


## 5. Visualisation Avancée

In [ ]:

# Carte mondiale de répartition géographique
df_geo = df[df['primary_fuel'].isin(fuels_main)].dropna(subset=['latitude','longitude'])
fuel_colors = {f: c for f, c in zip(fuels_main, sns.color_palette("tab10", len(fuels_main)))}

fig, ax = plt.subplots(figsize=(18, 9))
for fuel in fuels_main:
    sub = df_geo[df_geo['primary_fuel'] == fuel]
    ax.scatter(sub['longitude'], sub['latitude'],
               s=np.clip(sub['capacity_mw'] / 200, 1, 40),  # NumPy clip
               c=[fuel_colors[fuel]], alpha=0.35, label=fuel, linewidths=0)
ax.set_title("Carte mondiale des centrales électriques\n(taille des points ∝ capacité MW)", fontweight='bold', fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_xlim(-180,180); ax.set_ylim(-90,90)
ax.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.5)
ax.legend(loc='lower left', fontsize=8, markerscale=2)
ax.set_facecolor('#d6eaf8')
plt.tight_layout(); plt.show()


In [ ]:

# Heatmap de corrélation
gen_cols = [f'generation_gwh_{y}' for y in [2013,2014,2015,2016,2017]]
corr_df = df[['capacity_mw'] + gen_cols].dropna()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Corrélations : Capacité & Production annuelle (GWh)", fontweight='bold')
plt.tight_layout(); plt.show()


## 6. Opérations Matricielles – ACP (Analyse en Composantes Principales)

In [ ]:

# Matrice des caractéristiques moyennes par type de combustible
fuel_numeric = df_main.groupby('primary_fuel')[['capacity_mw','latitude','longitude']].mean().dropna()
X = fuel_numeric.values
feature_names = ['capacity_mw', 'latitude', 'longitude']

# 1. Standardisation (NumPy)
X_std = (X - X.mean(axis=0)) / X.std(axis=0)

# 2. Matrice de covariance
cov = np.cov(X_std.T)
print("Matrice de covariance :\n", np.round(cov, 3))

# 3. Décomposition propre
eigenvalues, eigenvectors = np.linalg.eig(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx].real
eigenvectors = eigenvectors[:, idx].real
explained = eigenvalues / eigenvalues.sum() * 100

print(f"\nValeurs propres : {np.round(eigenvalues, 3)}")
print(f"Variance expliquée (%) : {np.round(explained, 2)}")
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    contrib = {feature_names[j]: round(vec[j],3) for j in range(len(feature_names))}
    print(f"PC{i+1} ({explained[i]:.1f}%) – contributions : {contrib}")


In [ ]:

# Projection ACP manuelle (multiplication matricielle NumPy)
X_pca = X_std @ eigenvectors[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = sns.color_palette("tab10", len(fuel_numeric))
for i, fuel in enumerate(fuel_numeric.index):
    axes[0].scatter(X_pca[i,0], X_pca[i,1], s=140, color=colors[i], zorder=3, label=fuel)
    axes[0].annotate(fuel, (X_pca[i,0]+0.06, X_pca[i,1]), fontsize=8)
axes[0].axhline(0, color='gray', ls='--', lw=0.5)
axes[0].axvline(0, color='gray', ls='--', lw=0.5)
axes[0].set_title("ACP – Projection des types de combustible\n(capacité, latitude, longitude)", fontweight='bold')
axes[0].set_xlabel(f"PC1 ({explained[0]:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({explained[1]:.1f}% variance)")

# Variance expliquée par composante principale
axes[1].bar([f'PC{i+1}' for i in range(len(eigenvalues))], explained,
            color=sns.color_palette("pastel"))
axes[1].set_title("Variance expliquée par composante principale", fontweight='bold')
axes[1].set_ylabel("%")
plt.tight_layout(); plt.show()
# → PC1 (63%) : axe capacité + géographie.  PC2 (32%) : latitude seule.


## 7. Intégration NumPy + Pandas + Matplotlib

In [ ]:

# 7a – Filtrage complexe avec z-scores NumPy dans Pandas
mean_cap = df_main.groupby('primary_fuel')['capacity_mw'].transform('mean')
std_cap  = df_main.groupby('primary_fuel')['capacity_mw'].transform('std')
z_scores = (df_main['capacity_mw'] - mean_cap) / std_cap.replace(0, np.nan)

giants = df_main[z_scores > 2].copy()
print(f"Centrales géantes (z-score > 2 au sein de leur catégorie) : {len(giants)}")
print("\nTop 10 par capacité :")
giants[['name','country_long','primary_fuel','capacity_mw']].sort_values('capacity_mw', ascending=False).head(10)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Graphe violon avec percentiles NumPy ---
ax = axes[0]
order2 = df_main.groupby('primary_fuel')['capacity_mw'].median().sort_values(ascending=False).head(6).index.tolist()
data_v = [df_main[df_main['primary_fuel']==f]['capacity_mw'].dropna().values for f in order2]
vp = ax.violinplot(data_v, positions=range(len(order2)), showmedians=True, showextrema=False)
for i, data in enumerate(data_v):
    p25, p75 = np.percentile(data, [25, 75])   # ← NumPy percentile
    ax.plot([i-0.15, i+0.15], [p25, p25], 'k--', lw=1.2, alpha=0.7)
    ax.plot([i-0.15, i+0.15], [p75, p75], 'k--', lw=1.2, alpha=0.7)
for body in vp['bodies']: body.set_alpha(0.6)
ax.set_xticks(range(len(order2))); ax.set_xticklabels(order2, rotation=30)
ax.set_title("Violon + P25/P75 (NumPy) – Capacité MW", fontweight='bold')
ax.set_ylabel("Capacité (MW)")

# --- Courbe de Pareto avec cumsum NumPy ---
ax2 = axes[1]
top6_cap = df_main.groupby('primary_fuel')['capacity_mw'].sum().sort_values(ascending=False).head(6)
cumsum   = np.cumsum(top6_cap.values)   # ← NumPy cumsum
total    = cumsum[-1]
bars = ax2.bar(top6_cap.index, top6_cap.values, color=sns.color_palette("tab10",6), alpha=0.85)
ax2_r = ax2.twinx()
ax2_r.plot(range(len(top6_cap)), cumsum/total*100, 'ko-', lw=2, ms=7)
ax2_r.set_ylabel("% cumulatif (Pareto)")
ax2_r.set_ylim(0, 110)
for i, v in enumerate(cumsum/total*100):
    ax2_r.annotate(f"{v:.0f}%", (i, v+2), ha='center', fontsize=8)
ax2.set_title("Capacité totale & Courbe de Pareto (NumPy cumsum)", fontweight='bold')
ax2.set_ylabel("Capacité totale (MW)")
ax2.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
# → Charbon + Gaz + Hydro représentent ~76% de la capacité mondiale installée.


##README

# Analyse du paysage mondial des centrales électriques avec NumPy, Pandas et Matplotlib

## Introduction

L'objectif de cette étude est d'explorer la base de données mondiale des centrales électriques (*Global Power Plant Database*) afin de mieux comprendre la répartition géographique des installations, les sources d'énergie utilisées et les caractéristiques de production associées.

Pour réaliser cette analyse, trois bibliothèques fondamentales de l'écosystème Python ont été utilisées :

* **Pandas** pour le chargement, le nettoyage et la manipulation des données.
* **NumPy** pour les calculs numériques, les statistiques et les opérations matricielles.
* **Matplotlib** (associé à Seaborn) pour la visualisation des résultats.

L'ensemble de données contient près de 35 000 centrales électriques réparties dans plus de 160 pays.

---

# 1. Importation et nettoyage des données

## Chargement du dataset

Les données ont été importées à l'aide de Pandas :

```python
import pandas as pd

df = pd.read_csv("global_power_plant_database.csv")
```

## Traitement des valeurs manquantes

Une analyse des valeurs manquantes a montré que certaines colonnes relatives à la production annuelle et aux combustibles secondaires étaient fortement incomplètes.

Les principales actions réalisées ont été :

* Identification des valeurs nulles avec `isnull().sum()`.
* Conversion des colonnes numériques à l'aide de `pd.to_numeric()`.
* Suppression des doublons éventuels.
* Conservation des valeurs manquantes dans certaines colonnes non critiques afin d'éviter une perte excessive d'information.

---

# 2. Analyse exploratoire des données

## Structure générale

Le dataset contient :

* 34 936 centrales électriques
* 36 variables
* Des informations sur :

  * la localisation,
  * la capacité installée,
  * le combustible principal,
  * l'année de mise en service,
  * la production annuelle.

## Répartition par type de combustible

L'analyse révèle que les combustibles les plus représentés sont :

| Combustible | Nombre de centrales |
| ----------- | ------------------- |
| Solar       | 10 665              |
| Hydro       | 7 156               |
| Wind        | 5 344               |
| Gas         | 3 998               |
| Coal        | 2 330               |
| Oil         | 2 320               |

Cette distribution montre la forte présence des énergies renouvelables dans la base de données mondiale.

## Répartition géographique

Les pays comptant le plus grand nombre de centrales sont :

| Pays        | Nombre |
| ----------- | ------ |
| États-Unis  | 9 833  |
| Chine       | 4 235  |
| Royaume-Uni | 2 751  |
| Brésil      | 2 360  |
| France      | 2 155  |

Les États-Unis représentent à eux seuls près de 28 % des installations recensées.

---

# 3. Analyse statistique avec NumPy

Afin d'étudier la capacité des centrales selon leur source d'énergie, plusieurs indicateurs statistiques ont été calculés :

```python
np.mean()
np.median()
np.std()
```

## Capacité moyenne par combustible

| Combustible | Capacité moyenne (MW) |
| ----------- | --------------------- |
| Nuclear     | 2091.86               |
| Coal        | 843.58                |
| Gas         | 373.45                |
| Hydro       | 147.17                |
| Wind        | 49.22                 |
| Solar       | 17.66                 |

Les centrales nucléaires possèdent de loin la capacité moyenne la plus élevée.

Les centrales solaires et éoliennes sont beaucoup plus nombreuses mais généralement de plus petite taille.

---

# 4. Test d'hypothèse

Afin de vérifier si les capacités diffèrent réellement selon le combustible utilisé, une ANOVA à un facteur a été réalisée.

## Hypothèses

H₀ : toutes les capacités moyennes sont identiques.

H₁ : au moins une moyenne est différente.

```python
from scipy import stats

stats.f_oneway(*groupes)
```

## Résultat

* Statistique F : 1128.48
* p-value : < 0.001

## Conclusion

L'hypothèse nulle est rejetée.

Il existe une différence statistiquement significative entre les capacités moyennes des centrales selon leur type de combustible.

---

# 5. Analyse temporelle

L'année de mise en service des centrales a permis d'étudier l'évolution des technologies de production électrique.

## Évolution du nombre de centrales

Les données montrent une accélération importante des nouvelles installations à partir des années 2000.

Cette croissance est principalement portée par :

* le solaire,
* l'éolien,
* certaines installations hydroélectriques.

## Évolution des combustibles

Les centrales au charbon dominent historiquement les premières décennies.

À partir des années 2000, on observe :

* une forte progression du solaire ;
* une augmentation rapide de l'éolien ;
* une diversification progressive du mix énergétique mondial.

Cette évolution reflète les politiques de transition énergétique observées dans de nombreux pays.

---

# 6. Visualisations réalisées

Plusieurs graphiques ont été produits :

### Diagramme des combustibles

Permet d'identifier les principales sources d'énergie utilisées dans le monde.

### Répartition des pays

Montre les pays disposant du plus grand nombre d'installations.

### Courbe temporelle

Met en évidence la croissance des nouvelles centrales au cours du temps.

### Carte géographique

À partir des coordonnées GPS (latitude/longitude), les centrales ont été projetées afin d'observer leur répartition mondiale.

Les zones les plus denses sont :

* l'Amérique du Nord,
* l'Europe,
* l'Asie de l'Est.

---

# 7. Opérations matricielles et analyse avancée

NumPy a également été utilisé pour effectuer des opérations matricielles sur les variables quantitatives.

## Matrice de corrélation

Une matrice de corrélation a été construite à partir de :

* capacité (MW),
* latitude,
* longitude,
* année de mise en service.

```python
corr = np.corrcoef(matrix.T)
```

Cette matrice permet d'identifier les relations éventuelles entre les variables.

## Valeurs propres et vecteurs propres

Les valeurs propres et vecteurs propres ont été calculés avec :

```python
np.linalg.eig()
```

Ces outils sont particulièrement utiles pour :

* l'Analyse en Composantes Principales (ACP),
* la réduction de dimension,
* l'identification des facteurs expliquant le plus de variance dans les données.

Dans ce contexte, ils permettent de résumer efficacement les caractéristiques des centrales électriques mondiales.

---

# 8. Intégration de NumPy, Pandas et Matplotlib

Cette étude illustre parfaitement la complémentarité de ces trois bibliothèques.

## Pandas

Utilisé pour :

* charger les données ;
* nettoyer les colonnes ;
* regrouper les observations ;
* produire les statistiques descriptives.

## NumPy

Utilisé pour :

* les calculs statistiques ;
* les filtres complexes ;
* les opérations matricielles ;
* l'analyse de corrélation ;
* les valeurs propres et vecteurs propres.

## Matplotlib / Seaborn

Utilisés pour :

* les graphiques de distribution ;
* les diagrammes en barres ;
* les courbes temporelles ;
* les représentations géographiques.

---

# Conclusion

Cette analyse met en évidence plusieurs tendances importantes du secteur énergétique mondial.

Les principales observations sont :

* Les États-Unis dominent largement le nombre de centrales répertoriées.
* Les énergies renouvelables (solaire, hydroélectricité et éolien) représentent désormais la majorité des installations.
* Les centrales nucléaires possèdent les capacités de production les plus élevées.
* Les capacités varient significativement selon le combustible utilisé.
* Le développement des énergies renouvelables s'est fortement accéléré depuis le début du XXIᵉ siècle.

L'utilisation combinée de Pandas, NumPy et Matplotlib a permis de transformer un vaste ensemble de données en informations exploitables et de mieux comprendre l'évolution du paysage mondial de la production électrique.
